In [1]:
import ee
from ee_utils import init_gee
import hydrafloods as hf
import geemap


d:\Conda\envs\floodagent\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
init_gee()

{'initialized': True,
 'project_id': 'flood-agent',
 'key_file': 'D:\\2025\\.private-key.json',
 'mode': 'service_account'}

In [3]:
region = hf.country_bbox("Cambodia")
start_time = "2019-01-01"
end_time = "2019-07-01"

# get a Landsat 8 collection
lc8 = hf.Landsat8(region,start_time,end_time)
print(lc8)
# should look like
# HYDRAFloods Dataset:
# {'asset_id': 'LANDSAT/LC08/C01/T1_SR',
#  'end_time': '2019-07-01',
#  'name': 'Landsat8',
#  'region': [[[...], [...], [...], [...], [...]]],
#  'start_time': '2019-01-01'}

HYDRAFloods Dataset:
{'asset_id': 'LANDSAT/LC08/C02/T1_L2',
 'end_time': '2019-07-01',
 'name': 'Landsat8',
 'region': [[[...], [...], [...], [...], [...]]],
 'start_time': '2019-01-01'}


In [8]:
print(lc8.n_images)
# should equal 197

print(lc8.dates)
# should look something like
# ['2019-01-12 03:06:42.950',
#  '2019-01-28 03:06:38.990',
#  ... ,
# '2019-06-01 03:32:06.850']

# since `Dataset.collection` is a server side object we will just
# check that it is in fact a ee.ImageCollection object
print(isinstance(lc8.collection, ee.ImageCollection))
# should == True


207
['2019-01-12 03:06:42.951', '2019-01-28 03:06:38.989', '2019-02-13 03:06:36.944', '2019-03-01 03:06:33.071', '2019-03-17 03:06:27.542', '2019-04-02 03:06:24.371', '2019-04-18 03:06:18.636', '2019-05-04 03:06:18.722', '2019-05-20 03:06:29.369', '2019-06-05 03:06:37.599', '2019-06-21 03:06:43.585', '2019-01-12 03:07:06.855', '2019-01-28 03:07:02.897', '2019-02-13 03:07:00.848', '2019-03-01 03:06:56.979', '2019-03-17 03:06:51.446', '2019-04-02 03:06:48.274', '2019-04-18 03:06:42.536', '2019-05-04 03:06:42.630', '2019-05-20 03:06:53.281', '2019-06-05 03:07:01.511', '2019-06-21 03:07:07.493', '2019-01-12 03:07:30.759', '2019-01-28 03:07:26.801', '2019-02-13 03:07:24.752', '2019-03-01 03:07:20.883', '2019-03-17 03:07:15.354', '2019-04-02 03:07:12.182', '2019-04-18 03:07:06.444', '2019-05-04 03:07:06.542', '2019-05-20 03:07:17.189', '2019-06-05 03:07:25.415', '2019-06-21 03:07:31.401', '2019-01-12 03:07:54.662', '2019-01-28 03:07:50.709', '2019-02-13 03:07:48.664', '2019-03-01 03:07:44.79

In [4]:
region = hf.country_bbox("Cambodia")
start_time = "2019-01-01"
end_time = "2019-03-01"

# get a Landsat 8 collection
lc8 = hf.Landsat8(region,start_time,end_time)

water_index = lc8.apply_func(hf.mndwi)

water_ds = water_index.apply_func(hf.edge_otsu,
        initial_threshold=0,
        edge_buffer=300,
        scale=150,
        invert=True
)

index_img = water_index.collection.median()
water_img = water_ds.collection.mode()

water_img.getThumbURL({
    "min":-1,"max":1,
    "palette":"beige,white,lightblue,blue,darkblue",
    "dimensions":1500,
    "region":region
})

water_img.getThumbURL({
    "min":0,"max":1,
    "palette":"silver,navy",
    "dimensions":1500,
    "region":region
})

'https://earthengine.googleapis.com/v1/projects/flood-agent/thumbnails/4462d305aa38237fb9afc414ab395fe0-a52740725c8fda2915bed3b19643a3c2:getPixels'

In [15]:
map_id = water_img.getMapId({})
earth_engine_tile_url = map_id["tile_fetcher"].url_format
earth_engine_tile_url


'https://earthengine.googleapis.com/v1/projects/flood-agent/maps/c12416b3034ed9ff8debbddc5d219add-6d183a6af14c3c48f85c6e208bbc6cd3/tiles/{z}/{x}/{y}'

'https://earthengine.googleapis.com/v1/projects/flood-agent/maps/c12416b3034ed9ff8debbddc5d219add-2c2f38ec7ea980c9b4d8ca380e4fe977/tiles/{z}/{x}/{y}'

In [ ]:
import folium
map = folium.Map(location=[12.5657, 104.9910], zoom_start=6)
folium.TileLayer(
    tiles="https://earthengine.googleapis.com/v1/projects/flood-agent/maps/5139d5f224563e6a6e9f9e61f61ce4ac-a0178012269cb1835cb5dd80d40d1d3f/tiles/{z}/{x}/{y}",
    attr="Google Earth Engine",
    name="Water Mask",
    overlay=True,
    control=True
).add_to(map)

In [22]:
display(map)

In [16]:
import geemap

# 例如：XYZ 瓦片 URL（必须包含 {z}/{x}/{y}）
tile_url = earth_engine_tile_url

m = geemap.Map(center=[12.56, 104.99], zoom=6)
m.add_tile_layer(
    url=tile_url,
    name="My Tile Layer",
    attribution="My Tile Service",
    opacity=1.0
)
m

AttributeError: module 'ee.data' has no attribute '_credentials'

In [ ]:
!pip install -U geemap earthengine-api

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 1.4 MB/s eta 0:00:02
   ------------------------ --------------- 1.6/2.5 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 4.1 MB/s eta 0:00:00
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.7.20
    Uninstalling earthengine-api-1.7.20:
      Successfully uninstalled earthengine-api-1.7.20
  Attempting uninstall: geemap
    Found existing installation: geemap 0.35.3
    Uninstalling geemap-0.35.3:
      Successfully uninstalled geemap-0.35.3


: 

In [12]:
water_img.visualize({
    "min":0,"max":1,
    "palette":"silver,navy",
    "dimensions":1500,
    "region":region
})